# AGAR-RL: Autonomous Multi-Agent Deep Reinforcement Learning Pipeline

Ce notebook permet d'exécuter l'entraînement distribué par Deep Reinforcement Learning (PPO & Self-Play) directement depuis **VS Code** (via l'extension Google Colab ou votre kernel local) ou sur **Google Colab web**.

## 1. Détection de l'Environnement et Installation des Dépendances

In [ ]:
import os, sys

# 1. Vérification du répertoire de travail
if os.path.exists("src/training/train_colab.py"):
    print("✅ Exécution locale dans le workspace agario.")
else:
    print("🌐 Environnement distant Colab détecté. Clonage du repo...")
    !git clone https://github.com/Albin0903/agario.git
    %cd agario

# 2. Installation des dépendances
!pip install -q -r requirements.txt tensorboard

# 3. Vérification GPU CUDA
import torch
print(f"CUDA disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU actif : {torch.cuda.get_device_name(0)}")
else:
    print("Exécution sur CPU.")

## 2. Validation des Tests Unitaires (20 Tests)

In [ ]:
!pytest -v

## 3. Monitoring TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/tensorboard

## 4. Démarrer l'Entraînement PPO & Self-Play

In [ ]:
# Lance 16 environnements en parallèle avec 10 bots par arène et mise à jour du pool d'adversaires
!python src/training/train_colab.py \
    --n-envs 16 \
    --total-timesteps 1000000 \
    --batch-size 128 \
    --n-steps 2048 \
    --pool-interval 200000 \
    --device auto

## 5. Exporter la Politique vers ONNX (< 0.02 ms de latence CPU)

In [ ]:
!python src/inference/export_onnx.py \
    --model checkpoints/ppo/ppo_final.zip \
    --output models/model.onnx

## 6. Téléchargement du Modèle (si exécuté sur VM Colab distante)

In [ ]:
try:
    from google.colab import files
    files.download('models/model.onnx')
    files.download('checkpoints/ppo/ppo_final.zip')
    print("Téléchargement Colab initié.")
except ImportError:
    print("Fichiers sauvegardés localement dans models/ et checkpoints/.")